# Validate Platform Readiness

Assertion-based validation of the post-bootstrap platform contract. Confirms
that all infrastructure prerequisites are in place for the lakeLoom Databricks
App and QR-pair onboarding flow.

**Validates:**
1. SQL warehouse session is operating in the expected catalog/schema
2. Unity Catalog schema exists
3. Managed `session_audio` volume exists and is MANAGED type
4. Managed `screenshots` volume exists and is MANAGED type
5. Managed `documents` volume exists and is MANAGED type
6. ZeroBus SPN has READ_VOLUME + WRITE_VOLUME on all three managed volumes
7. Bronze `transcript_events_raw` table exists (created by upstream DDL task)

This notebook runs as the **final task** in the `platform_bootstrap` job.

In [0]:
%sql
DECLARE OR REPLACE VARIABLE catalog_use STRING;
DECLARE OR REPLACE VARIABLE schema_use STRING;

SET VARIABLE catalog_use = :catalog_use;
SET VARIABLE schema_use = :schema_use;

USE IDENTIFIER(catalog_use || '.' || schema_use);

SELECT
  current_timestamp() AS validated_at,
  current_catalog() AS active_catalog,
  current_schema() AS active_schema,
  current_user() AS executed_as,
  :secret_scope_name AS secret_scope_name,
  :client_id_dbs_key AS client_id_dbs_key,
  :client_secret_dbs_key AS client_secret_dbs_key,
  :zerobus_stream_pool_size AS zerobus_stream_pool_size,
  catalog_use || '.' || schema_use || '.transcript_events_raw' AS expected_target_table_name,
  '/Volumes/' || catalog_use || '/' || schema_use || '/session_audio' AS expected_volume_path;

In [0]:
%sql
SELECT
  assert_true(
    current_catalog() = catalog_use,
    'Active catalog mismatch. Expected ' || catalog_use || ' but session is using ' || current_catalog()
  ) AS catalog_context_valid,
  assert_true(
    current_schema() = schema_use,
    'Active schema mismatch. Expected ' || schema_use || ' but session is using ' || current_schema()
  ) AS schema_context_valid;

In [0]:
%sql
WITH schema_match AS (
  SELECT *
  FROM system.information_schema.schemata
  WHERE catalog_name = catalog_use
    AND schema_name = schema_use
)
SELECT
  assert_true(
    COUNT(*) = 1,
    'Required schema not found: ' || catalog_use || '.' || schema_use
  ) AS schema_exists,
  MAX(schema_owner) AS schema_owner,
  MAX(created) AS schema_created_at,
  MAX(created_by) AS schema_created_by,
  MAX(comment) AS schema_comment
FROM schema_match;

In [0]:
%sql
WITH volume_match AS (
  SELECT *
  FROM system.information_schema.volumes
  WHERE volume_catalog = catalog_use
    AND volume_schema = schema_use
    AND volume_name = 'session_audio'
),
volume_summary AS (
  SELECT
    COUNT(*) AS matching_volume_count,
    COALESCE(MAX(CASE WHEN volume_type = 'MANAGED' THEN 1 ELSE 0 END), 0) AS has_managed_volume,
    MAX(volume_type) AS detected_volume_type,
    MAX(storage_location) AS storage_location,
    MAX(comment) AS volume_comment,
    MAX(created) AS volume_created_at,
    MAX(created_by) AS volume_created_by
  FROM volume_match
)
SELECT
  assert_true(
    matching_volume_count = 1,
    'Required volume not found: ' || catalog_use || '.' || schema_use || '.session_audio'
  ) AS session_audio_volume_exists,
  assert_true(
    has_managed_volume = 1,
    'Volume exists but is not MANAGED: ' || catalog_use || '.' || schema_use || '.session_audio'
  ) AS session_audio_volume_is_managed,
  detected_volume_type AS volume_type,
  storage_location,
  volume_comment,
  volume_created_at,
  volume_created_by,
  '/Volumes/' || catalog_use || '/' || schema_use || '/session_audio' AS expected_volume_files_api_path
FROM volume_summary;

In [0]:
WITH volume_match AS (
  SELECT *
  FROM system.information_schema.volumes
  WHERE volume_catalog = catalog_use
    AND volume_schema = schema_use
    AND volume_name = 'screenshots'
),
volume_summary AS (
  SELECT
    COUNT(*) AS matching_volume_count,
    COALESCE(MAX(CASE WHEN volume_type = 'MANAGED' THEN 1 ELSE 0 END), 0) AS has_managed_volume,
    MAX(volume_type) AS detected_volume_type,
    MAX(storage_location) AS storage_location,
    MAX(comment) AS volume_comment,
    MAX(created) AS volume_created_at,
    MAX(created_by) AS volume_created_by
  FROM volume_match
)
SELECT
  assert_true(
    matching_volume_count = 1,
    'Required volume not found: ' || catalog_use || '.' || schema_use || '.screenshots'
  ) AS screenshots_volume_exists,
  assert_true(
    has_managed_volume = 1,
    'Volume exists but is not MANAGED: ' || catalog_use || '.' || schema_use || '.screenshots'
  ) AS screenshots_volume_is_managed,
  detected_volume_type AS volume_type,
  storage_location,
  volume_comment,
  volume_created_at,
  volume_created_by,
  '/Volumes/' || catalog_use || '/' || schema_use || '/screenshots' AS expected_volume_files_api_path
FROM volume_summary;

In [0]:
WITH volume_match AS (
  SELECT *
  FROM system.information_schema.volumes
  WHERE volume_catalog = catalog_use
    AND volume_schema = schema_use
    AND volume_name = 'documents'
),
volume_summary AS (
  SELECT
    COUNT(*) AS matching_volume_count,
    COALESCE(MAX(CASE WHEN volume_type = 'MANAGED' THEN 1 ELSE 0 END), 0) AS has_managed_volume,
    MAX(volume_type) AS detected_volume_type,
    MAX(storage_location) AS storage_location,
    MAX(comment) AS volume_comment,
    MAX(created) AS volume_created_at,
    MAX(created_by) AS volume_created_by
  FROM volume_match
)
SELECT
  assert_true(
    matching_volume_count = 1,
    'Required volume not found: ' || catalog_use || '.' || schema_use || '.documents'
  ) AS documents_volume_exists,
  assert_true(
    has_managed_volume = 1,
    'Volume exists but is not MANAGED: ' || catalog_use || '.' || schema_use || '.documents'
  ) AS documents_volume_is_managed,
  detected_volume_type AS volume_type,
  storage_location,
  volume_comment,
  volume_created_at,
  volume_created_by,
  '/Volumes/' || catalog_use || '/' || schema_use || '/documents' AS expected_volume_files_api_path
FROM volume_summary;

In [0]:
-- Receive spn_application_id from upstream ensure_service_principal task.
DECLARE OR REPLACE VARIABLE spn_application_id STRING;
SET VARIABLE spn_application_id = :spn_application_id;

SELECT spn_application_id AS validating_volume_grants_for;

In [0]:
WITH vol_grants AS (
  SELECT *
  FROM information_schema.volume_privileges
  WHERE volume_schema = schema_use
    AND volume_name = 'session_audio'
    AND grantee LIKE '%' || spn_application_id || '%'
)
SELECT
  assert_true(
    (SELECT COUNT(*) FROM vol_grants WHERE privilege_type = 'READ_VOLUME') >= 1,
    'Missing READ_VOLUME on session_audio for SPN ' || spn_application_id
  ) AS session_audio_read_volume,
  assert_true(
    (SELECT COUNT(*) FROM vol_grants WHERE privilege_type = 'WRITE_VOLUME') >= 1,
    'Missing WRITE_VOLUME on session_audio for SPN ' || spn_application_id
  ) AS session_audio_write_volume;

In [0]:
WITH vol_grants AS (
  SELECT *
  FROM information_schema.volume_privileges
  WHERE volume_schema = schema_use
    AND volume_name = 'screenshots'
    AND grantee LIKE '%' || spn_application_id || '%'
)
SELECT
  assert_true(
    (SELECT COUNT(*) FROM vol_grants WHERE privilege_type = 'READ_VOLUME') >= 1,
    'Missing READ_VOLUME on screenshots for SPN ' || spn_application_id
  ) AS screenshots_read_volume,
  assert_true(
    (SELECT COUNT(*) FROM vol_grants WHERE privilege_type = 'WRITE_VOLUME') >= 1,
    'Missing WRITE_VOLUME on screenshots for SPN ' || spn_application_id
  ) AS screenshots_write_volume;

In [0]:
WITH vol_grants AS (
  SELECT *
  FROM information_schema.volume_privileges
  WHERE volume_schema = schema_use
    AND volume_name = 'documents'
    AND grantee LIKE '%' || spn_application_id || '%'
)
SELECT
  assert_true(
    (SELECT COUNT(*) FROM vol_grants WHERE privilege_type = 'READ_VOLUME') >= 1,
    'Missing READ_VOLUME on documents for SPN ' || spn_application_id
  ) AS documents_read_volume,
  assert_true(
    (SELECT COUNT(*) FROM vol_grants WHERE privilege_type = 'WRITE_VOLUME') >= 1,
    'Missing WRITE_VOLUME on documents for SPN ' || spn_application_id
  ) AS documents_write_volume;

In [0]:
%sql
WITH target_table_match AS (
  SELECT *
  FROM system.information_schema.tables
  WHERE table_catalog = catalog_use
    AND table_schema = schema_use
    AND table_name = 'transcript_events_raw'
)
SELECT
  assert_true(
    COUNT(*) = 1,
    'Required target table not found: ' || catalog_use || '.' || schema_use || '.transcript_events_raw'
  ) AS transcript_events_raw_exists,
  MAX(table_type) AS target_table_type,
  MAX(data_source_format) AS data_source_format,
  MAX(storage_path) AS storage_path
FROM target_table_match;

In [0]:
SELECT
  'platform_bootstrap_validation_complete' AS check_name,
  'schema, managed volumes, volume grants, and transcript_events_raw assertions passed' AS status,
  catalog_use || '.' || schema_use AS validated_schema,
  spn_application_id AS validated_spn,
  '/Volumes/' || catalog_use || '/' || schema_use || '/session_audio' AS validated_audio_volume_path,
  '/Volumes/' || catalog_use || '/' || schema_use || '/screenshots' AS validated_screenshots_volume_path,
  '/Volumes/' || catalog_use || '/' || schema_use || '/documents' AS validated_documents_volume_path,
  catalog_use || '.' || schema_use || '.transcript_events_raw' AS validated_target_table_name,
  current_timestamp() AS completed_at;